# 13 Contested Regions Weekly Presentation Analysis


## Core definition

A **contested region** is a contiguous protein fragment where calibrated disorder predictors disagree strongly about residue-level disorder.

Working convention used throughout this project:

- higher score means more disorder;
- CheZOD is negated to match this direction;
- pLDDT is converted to `1 - pLDDT / 100`;
- predictors are calibrated before comparing spread, e.g. z-score or quantile normalization.

First-pass region score:

```text
residue_disagreement_i = robust_spread(z_1i, ..., z_ki)
window_score = mean(residue_disagreement_i over a 30-residue window)
```

For experiment prioritization, use:

```text
priority = predictor_disagreement * boundary_uncertainty * experiment_suitability
```

In [ ]:
from pathlib import Path
import json
import math
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RESULTS = ROOT / 'results'
OUT = RESULTS / 'weekly_contested_regions'
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 220,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SLIDE = RESULTS / 'slide_enhanced_ceiling'
HUMAN = RESULTS / 'human_proteome_annotation_ceiling'
COMPARE = RESULTS / 'compare_predictors_with_seth_and_iupred3_and_adopt_and_metapredict'
ANNOT = RESULTS / 'annotation_ceiling'
ANNOT_MMSEQS = RESULTS / 'annotation_ceiling_mmseqs'

def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, **kwargs)

def savefig(name):
    path = OUT / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'saved: {path.relative_to(ROOT)}')
    return path

## 1. Check available evidence

These are the inputs this notebook expects. Missing files mean the corresponding plot cell should be skipped or regenerated by upstream scripts.

In [ ]:
required = {
    'contested definition note': ROOT / 'docs' / 'contested_regions.md',
    'human overlap summary': HUMAN / 'human_proteome_overlap_summary.csv',
    'human overlap details': HUMAN / 'human_proteome_overlap_details.csv',
    'same vs cross concept': SLIDE / 'same_vs_cross_concept_summary.csv',
    'family agreement': SLIDE / 'annotation_family_agreement_summary.csv',
    'region-level agreement': SLIDE / 'region_level_agreement.csv',
    'primary biological compatibility': SLIDE / 'primary_ceiling_biological_compatibility.csv',
    'exact vs mmseqs80': SLIDE / 'exact_vs_mmseqs80_primary_ceiling.csv',
    'top contested regions figure': COMPARE / 'top_contested_regions.png',
    'PDBFlex outlier figure': COMPARE / 'contested_regions_pdbflex_outlier_profile.png',
    'predictor pairwise bars': COMPARE / 'predictor_pairwise_spearman_bars.png',
}

availability = pd.DataFrame([
    {'artifact': name, 'exists': path.exists(), 'path': str(path.relative_to(ROOT))}
    for name, path in required.items()
])
availability

## 2. Human-proteome annotation coverage

This plot answers: if we rank contested regions in the reviewed human proteome, how much existing UdonPred-style annotation can we use for direct validation?

In [ ]:
human_summary = read_csv(HUMAN / 'human_proteome_overlap_summary.csv')
coverage = human_summary.copy()
coverage['matched_protein_pct'] = 100 * coverage['protein_overlap_fraction']
coverage['matched_valid_residue_pct'] = 100 * coverage['valid_annotated_residue_overlap_fraction']
coverage = coverage.sort_values('matched_valid_residue_pct', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4.8))
y = np.arange(len(coverage))
ax.barh(y - 0.18, coverage['matched_protein_pct'], height=0.34, label='matched proteins', color='#4C78A8')
ax.barh(y + 0.18, coverage['matched_valid_residue_pct'], height=0.34, label='matched valid residues', color='#F58518')
ax.set_yticks(y)
ax.set_yticklabels(coverage['dataset'])
ax.set_xlabel('Human proteome overlap (%)')
ax.set_title('Existing labels cover only a small part of the human proteome')
ax.legend(frameon=False, loc='lower right')
for i, (_, row) in enumerate(coverage.iterrows()):
    ax.text(row['matched_valid_residue_pct'] + 0.7, i + 0.18, f"{row['matched_valid_residue_pct']:.1f}%", va='center', fontsize=9)
savefig('human_annotation_coverage.png')
plt.show()

coverage[['dataset', 'n_matched_human_proteins', 'matched_protein_pct', 'matched_valid_residue_pct']].sort_values('matched_valid_residue_pct', ascending=False)

**Presentation takeaway:** pLDDT and DisProt have the broadest exact-sequence overlap with the reviewed human proteome in the existing evaluation sets. NMR chemical-shift annotations have very low direct human coverage, which makes disagreement-guided experimental selection relevant.

## 3. Annotation concepts are not interchangeable

Contested regions matter because disagreement is not just model noise. The project brief says different evidence sources measure different facets of disorder. The branch already has an annotation-ceiling analysis that makes this visible.

In [ ]:
compat = read_csv(SLIDE / 'primary_ceiling_biological_compatibility.csv')
compat = compat[(compat['primary_metric'].isin(['spearman', 'auroc'])) & compat['value'].notna()].copy()
compat['display_pair'] = compat['pair'] + ' (' + compat['primary_metric'] + ')'
compat = compat.sort_values('value')

colors = {
    'low': '#E45756',
    'medium': '#F58518',
    'high': '#54A24B',
    'unknown/no-overlap': '#BAB0AC',
}
bar_colors = compat['expected_biological_similarity'].map(colors).fillna('#4C78A8')

fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(compat))))
ax.barh(compat['display_pair'], compat['value'], color=bar_colors)
ax.set_xlabel('Primary annotation agreement ceiling')
ax.set_title('Real annotation agreement depends on the biological concept')
ax.axvline(0.5, color='black', lw=1, alpha=0.35)
for y_pos, value in enumerate(compat['value']):
    ax.text(value + 0.015, y_pos, f'{value:.2f}', va='center', fontsize=8)
savefig('annotation_concept_compatibility.png')
plt.show()

compat[['pair', 'family_a', 'family_b', 'expected_biological_similarity', 'primary_metric', 'value', 'n_residues_compared']].head(10)

In [ ]:
same_cross = read_csv(SLIDE / 'same_vs_cross_concept_summary.csv')
order = ['same annotation family', 'related disorder concept', 'different concept: flexibility vs disorder/proxy', 'unknown or weakly related']
same_cross['concept_relation'] = pd.Categorical(same_cross['concept_relation'], categories=order, ordered=True)
same_cross = same_cross.sort_values('concept_relation')

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(same_cross['concept_relation'].astype(str), same_cross['median_ceiling'], color='#4C78A8')
ax.set_ylabel('Median annotation ceiling')
ax.set_title('Median agreement drops when annotation concepts diverge')
ax.set_ylim(0, max(1.0, same_cross['median_ceiling'].max() + 0.1))
ax.tick_params(axis='x', rotation=25)
for i, row in same_cross.reset_index(drop=True).iterrows():
    ax.text(i, row['median_ceiling'] + 0.025, f"{row['median_ceiling']:.2f}\nn={int(row['n_pairs'])}", ha='center', va='bottom', fontsize=9)
savefig('same_vs_cross_concept_summary.png')
plt.show()

same_cross

**Presentation takeaway:** model disagreement should be interpreted by concept. PDBFlex/flexibility can be a valid signal but it is not the same biological target as NMR chemical-shift disorder or curated DisProt disorder.

## 4. Exact matching vs. MMseqs matching

This plot is useful for explaining why exact-overlap ceilings are conservative: local sequence matching can add comparable residues, but it may also change the apparent agreement.

In [ ]:
exact_mm = read_csv(SLIDE / 'exact_vs_mmseqs80_primary_ceiling.csv')
exact_mm = exact_mm[exact_mm['exact_value'].notna() & exact_mm['mmseqs80_value'].notna()].copy()
exact_mm['residue_gain_label'] = exact_mm['residue_gain'].map(lambda x: f'{int(x):+d}')
exact_mm = exact_mm.sort_values('delta_mmseqs80_minus_exact')

fig, ax = plt.subplots(figsize=(10, max(4.5, 0.38 * len(exact_mm))))
colors = np.where(exact_mm['delta_mmseqs80_minus_exact'] >= 0, '#54A24B', '#E45756')
ax.barh(exact_mm['pair'], exact_mm['delta_mmseqs80_minus_exact'], color=colors)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('MMseqs80 ceiling minus exact-match ceiling')
ax.set_title('Local sequence matches can change estimated annotation agreement')
for y_pos, row in enumerate(exact_mm.itertuples()):
    ax.text(row.delta_mmseqs80_minus_exact + (0.02 if row.delta_mmseqs80_minus_exact >= 0 else -0.02),
            y_pos,
            f'{row.delta_mmseqs80_minus_exact:+.2f} / residues {int(row.residue_gain):+d}',
            va='center',
            ha='left' if row.delta_mmseqs80_minus_exact >= 0 else 'right',
            fontsize=8)
savefig('exact_vs_mmseqs80_delta.png')
plt.show()

exact_mm[['pair', 'exact_value', 'mmseqs80_value', 'delta_mmseqs80_minus_exact', 'exact_residues', 'mmseqs80_residues', 'concept_relation']]

## 5. Region-level agreement

For presentation, residue-level correlations are abstract. Region-level overlap metrics are easier to explain: do two annotation sources mark the same stretches as disordered?

In [ ]:
region = read_csv(SLIDE / 'region_level_agreement.csv')
region = region.sort_values('region_f1', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4.5, 0.36 * len(region))))
ax.barh(region['pair'], region['region_f1'], color='#72B7B2')
ax.set_xlabel('Region-level F1')
ax.set_title('Some annotation pairs agree on regions even when residue scores differ')
ax.set_xlim(0, 1.05)
for y_pos, row in enumerate(region.itertuples()):
    ax.text(row.region_f1 + 0.015, y_pos, f'{row.region_f1:.2f}', va='center', fontsize=8)
savefig('region_level_agreement.png')
plt.show()

region[['pair', 'family_a', 'family_b', 'n_residues', 'n_regions_a', 'n_regions_b', 'region_precision', 'region_recall', 'region_f1', 'residue_jaccard']]

## 6. Existing predictor-disagreement visuals

The following figures already exist in the branch and are directly usable in weekly slides.

In [ ]:
figure_paths = [
    COMPARE / 'predictor_pairwise_spearman_bars.png',
    COMPARE / 'top_contested_regions.png',
    COMPARE / 'contested_regions_pdbflex_outlier_profile.png',
    COMPARE / 'predictor_vs_udonpred_consensus.png',
    COMPARE / 'udon_predictor_annotation_residue_gap_exact_vs_mmseqs.png',
]

for path in figure_paths:
    if path.exists():
        display(Markdown(f'### `{path.relative_to(ROOT)}`'))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f'**Missing:** `{path.relative_to(ROOT)}`'))

## 7. Presentation-ready contested-region ranking

The branch already has `top_contested_regions.png`; the raw table was not checked into the repo. For weekly slides, this cell recreates a compact table from the visible top entries in that figure and saves a cleaner bar chart.

If a raw table becomes available later, replace the literal table below with a `read_csv(...)` call.

In [ ]:
top_regions = pd.DataFrame([
    ('P48594', 331, 360, 2.17),
    ('P01011', 361, 390, 2.09),
    ('P29508', 331, 360, 2.06),
    ('Q9UIV8', 331, 360, 1.99),
    ('P01009', 361, 390, 1.99),
    ('Q86WD7', 361, 390, 1.97),
    ('O75830', 331, 360, 1.94),
    ('P29622', 361, 390, 1.94),
    ('Q9UK55', 391, 420, 1.91),
    ('O95140', 601, 630, 1.89),
    ('O75635', 331, 360, 1.88),
    ('Q96P15', 331, 360, 1.85),
    ('O95236', 181, 210, 1.82),
    ('P05155', 451, 480, 1.82),
    ('Q8IW75', 361, 390, 1.81),
], columns=['accession', 'start', 'end', 'mean_disagreement'])
top_regions['region'] = top_regions['accession'] + ':' + top_regions['start'].astype(str) + '-' + top_regions['end'].astype(str)

top_regions = top_regions.sort_values('mean_disagreement', ascending=True)
fig, ax = plt.subplots(figsize=(9.5, 6.2))
ax.barh(top_regions['region'], top_regions['mean_disagreement'], color='#C44E52')
ax.set_xlabel('Average predictor disagreement in 30-residue window')
ax.set_title('Top contested human-proteome regions')
for y_pos, value in enumerate(top_regions['mean_disagreement']):
    ax.text(value + 0.025, y_pos, f'{value:.2f}', va='center', fontsize=9)
savefig('top_contested_regions_replotted.png')
plt.show()

top_regions.sort_values('mean_disagreement', ascending=False)

**Presentation takeaway:** the highest-ranked windows cluster around 30-residue segments with very large normalized predictor spread. The existing outlier plot suggests many are driven by a high PDBFlex-head score relative to the median predictor, so these are likely flexibility-vs-disorder conflicts.

## 8. Proposed contested-region score visual

This is a conceptual slide figure: predictor disagreement is necessary but not sufficient. The most useful experimental candidates should also be near the decision boundary and practical for NMR follow-up.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axis('off')

boxes = [
    (0.05, 0.54, 0.23, 0.28, 'Predictor\ndisagreement', 'robust spread across\ncalibrated predictors'),
    (0.39, 0.54, 0.23, 0.28, 'Boundary\nuncertainty', 'consensus near\norder/disorder threshold'),
    (0.73, 0.54, 0.23, 0.28, 'Experiment\nsuitability', 'fragment length, solubility,\nno obvious artifacts'),
    (0.32, 0.10, 0.36, 0.25, 'Ranked contested\nregions', 'highest expected information\ngain for new annotation'),
]

for x, y, w, h, title, subtitle in boxes:
    rect = plt.Rectangle((x, y), w, h, transform=ax.transAxes, facecolor='#F2F2F2', edgecolor='#4C78A8', linewidth=1.8)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.66, title, ha='center', va='center', transform=ax.transAxes, fontsize=14, weight='bold')
    ax.text(x + w/2, y + h*0.28, subtitle, ha='center', va='center', transform=ax.transAxes, fontsize=10)

for x0, x1 in [(0.28, 0.39), (0.62, 0.73)]:
    ax.annotate('', xy=(x1, 0.68), xytext=(x0, 0.68), xycoords=ax.transAxes,
                arrowprops=dict(arrowstyle='->', lw=2, color='#666666'))
for x0 in [0.165, 0.505, 0.845]:
    ax.annotate('', xy=(0.50, 0.35), xytext=(x0, 0.54), xycoords=ax.transAxes,
                arrowprops=dict(arrowstyle='->', lw=1.8, color='#666666'))

ax.text(0.50, 0.92, 'Contested-region priority = disagreement × uncertainty × suitability',
        ha='center', transform=ax.transAxes, fontsize=15, weight='bold')
savefig('contested_region_scoring_concept.png')
plt.show()

## 9. Candidate slide sequence

1. **Problem:** disorder annotations measure different biology, so labels disagree.
2. **Evidence:** annotation-ceiling plot showing concept-dependent agreement.
3. **Coverage gap:** human-proteome exact overlap is limited, especially for NMR labels.
4. **Predictor disagreement:** pairwise predictor agreement plot, highlighting PDBFlex as weakly aligned with disorder predictors.
5. **Contested regions:** top 30-residue windows by calibrated predictor spread.
6. **Interpretation:** many top windows are PDBFlex-high outliers, suggesting flexibility-vs-disorder conflict.
7. **Next step:** rank windows by disagreement × boundary uncertainty × NMR suitability, then validate retrospectively against later BMRB coverage.

## 11. Full contested-region pipeline output

The first version of this notebook used existing PNGs. The repository now has a reusable script:

```bash
python3 scripts/analyze_contested_regions.py --target-dataset plddt --output-dir results/contested_regions/plddt
```

It creates the actual window-level table requested for follow-up analysis:

- `contested_windows.csv`: all scored windows;
- `top_contested_windows.csv`: ranked experimental candidates;
- `disagreement_type_summary.csv`: PDBFlex-high, pLDDT-outlier, NMR-vs-curated, broad-spread classes;
- `feature_enrichment_top_vs_background.csv`: sequence feature enrichment;
- `validation_summary.csv`: sanity checks and label-boundary correlation;
- `case_studies/*.png`: predictor tracks for top-ranked regions.

The current checked-in raw human predictor outputs are not available, so the verified run below uses the existing UdonPred cross-head predictions on the pLDDT target set. The same script supports future human-proteome outputs with `--layout generic --prediction-root <folder>`.

In [ ]:
CONTESTED = RESULTS / 'contested_regions' / 'plddt'
for required_file in [
    'contested_windows.csv',
    'top_contested_windows.csv',
    'disagreement_type_summary.csv',
    'feature_enrichment_top_vs_background.csv',
    'validation_summary.csv',
]:
    path = CONTESTED / required_file
    print(f'{required_file}:', 'ok' if path.exists() else 'missing')

### Ranked candidate table

This is the core table for deciding which regions to discuss as experimental candidates.

In [ ]:
contested_windows = read_csv(CONTESTED / 'contested_windows.csv')
top_contested = read_csv(CONTESTED / 'top_contested_windows.csv')
cols = [
    'protein_id', 'start', 'end', 'priority_score', 'mean_disagreement',
    'boundary_uncertainty', 'nmr_suitability', 'disagreement_type',
    'top_predictor', 'bottom_predictor', 'top_minus_bottom_z',
    'hydropathy', 'net_charge', 'frac_pro_gly', 'low_complexity',
    'has_human_overlap', 'human_accessions', 'human_genes',
]
top_contested[[col for col in cols if col in top_contested.columns]].head(15)

### Disagreement-type classes

This turns a variance score into a biological interpretation: flexibility-vs-disorder outliers, pLDDT uncertainty outliers, NMR-vs-curated conflicts, broad multi-model spread, and general spread.

In [ ]:
disagreement_type_summary = read_csv(CONTESTED / 'disagreement_type_summary.csv')
disagreement_type_summary

In [ ]:
for name in [
    'top_contested_windows.png',
    'disagreement_type_counts.png',
    'feature_enrichment_top_vs_background.png',
    'disagreement_vs_boundary_uncertainty.png',
]:
    path = CONTESTED / name
    if path.exists():
        display(Markdown(f'### `{path.relative_to(ROOT)}`'))
        display(Image(filename=str(path)))

### Feature enrichment and validation

This checks whether top contested windows have distinctive sequence properties and whether disagreement relates to target-label boundary ambiguity.

In [ ]:
feature_enrichment = read_csv(CONTESTED / 'feature_enrichment_top_vs_background.csv')
validation = read_csv(CONTESTED / 'validation_summary.csv')
display(feature_enrichment)
display(validation)

### Case-study panels

Each panel shows calibrated predictor tracks around a top-ranked region. These are the most useful weekly-slide visuals because they show exactly which predictors disagree and where.

In [ ]:
case_dir = CONTESTED / 'case_studies'
for path in sorted(case_dir.glob('*.png'))[:5]:
    display(Markdown(f'### `{path.relative_to(ROOT)}`'))
    display(Image(filename=str(path)))

## 12. Immediate follow-up after raw human predictions exist

Run the same pipeline on human-proteome predictor outputs laid out as:

```text
results/human_predictor_outputs/
  trizod/*.caid
  chezod/*.caid
  softdis/*.caid
  pdbflex/*.caid
  atlas/*.caid
  plddt/*.caid
  disprot/*.caid
  seth/*.caid
  iupred3/*.caid
  adopt/*.caid
  metapredict/*.caid
```

Then execute:

```bash
python3 scripts/analyze_contested_regions.py \
  --layout generic \
  --prediction-root results/human_predictor_outputs \
  --target-dataset human \
  --output-dir results/contested_regions/human
```

That will produce the same ranked window table, disagreement classes, feature enrichment, and case-study plots for the actual reviewed human proteome.

## 13. Lecture-derived biological controls

The lecture slides add several biological controls that make the contested-region analysis more defensible:

- disorder can mean long no-regular-secondary-structure regions, B-value/flexibility, contact-deprived regions, NMR chemical-shift disorder, or embedding-based disorder;
- B-values/flexibility are not directly equivalent to intrinsic disorder, so PDBFlex-high windows need their own interpretation;
- transmembrane helices and signal peptides are hydrophobic, segment-like cases that can look unusual to disorder predictors and are poor NMR candidates;
- low pLDDT is useful as a disorder proxy, but it is still an AlphaFold confidence score;
- MoRE-like regions are interesting because some disordered regions become structured upon binding.

The script now adds these lecture-derived columns to `contested_windows.csv`: `tmh_like_score`, `signal_peptide_like_score`, `coil_like_score`, `nors_like_score`, `possible_more_score`, `concept_family_spread_z`, and `plddt_confidence_class`.

In [ ]:
lecture_summary = read_csv(CONTESTED / 'lecture_extension_summary.csv')
lecture_summary

In [ ]:
for name in [
    'lecture_controls_top_vs_all.png',
    'lecture_controls_top_windows_profile.png',
]:
    path = CONTESTED / name
    if path.exists():
        display(Markdown(f'### `{path.relative_to(ROOT)}`'))
        display(Image(filename=str(path)))

### Lecture-informed interpretation table

This table shows the new biological controls next to the top-ranked windows. Use it to decide whether a candidate is a clean experimental target, a possible MoRE/binding motif, or likely a membrane/signal-peptide artifact.

In [ ]:
lecture_cols = [
    'protein_id', 'start', 'end', 'priority_score', 'disagreement_type',
    'concept_family_spread_z', 'possible_more_score', 'tmh_like_score',
    'signal_peptide_like_score', 'coil_like_score', 'nors_like_score',
    'plddt_confidence_class', 'percent_hydrophobic', 'net_charge',
    'low_complexity', 'top_predictor', 'bottom_predictor',
]
top_contested[[col for col in lecture_cols if col in top_contested.columns]].head(20)

**Presentation takeaway:** top-priority windows are enriched for concept-family spread, meaning the strongest candidates are not only single-predictor noise. TMH/signal-like scores are lower in the top-priority set, which is good: the ranking is not dominated by obvious membrane artifacts. The MoRE score highlights a smaller set of high-disagreement, non-membrane, moderately disordered candidates that may be worth case-study discussion.

## 14. PDBFlex ablation: what if we remove it completely?

Because PDBFlex repeatedly appears as an outlier, we ran a direct ablation: same target set, same windows, same scoring pipeline, but with the `pdbflex` predictor removed before calibration and scoring.

Command:

```bash
python3 scripts/analyze_contested_regions.py \
  --target-dataset plddt \
  --exclude-predictors pdbflex \
  --output-dir results/contested_regions/plddt_no_pdbflex

python3 scripts/compare_contested_ablation.py \
  --baseline-dir results/contested_regions/plddt \
  --ablation-dir results/contested_regions/plddt_no_pdbflex \
  --output-dir results/contested_regions/pdbflex_ablation_comparison
```

In [ ]:
ABLATION = RESULTS / 'contested_regions' / 'pdbflex_ablation_comparison'
score_summary = pd.read_csv(ABLATION / 'score_summary.csv', index_col=0)
top_overlap = pd.read_csv(ABLATION / 'top_region_overlap_summary.csv')
type_comparison = pd.read_csv(ABLATION / 'disagreement_type_count_comparison.csv')
display(score_summary)
display(top_overlap)
display(type_comparison)

In [ ]:
for name in [
    'score_summary_comparison.png',
    'rank_stability_scatter.png',
    'disagreement_type_comparison.png',
]:
    path = ABLATION / name
    if path.exists():
        display(Markdown(f'### `{path.relative_to(ROOT)}`'))
        display(Image(filename=str(path)))

**Interpretation:** removing PDBFlex lowers the median disagreement score and especially the concept-family spread. The global ranking remains correlated, but the very top candidates change substantially: only 13 of the top 25 windows are shared. This means PDBFlex is not only adding random noise; it creates a distinct flexibility-vs-disorder axis. If the scientific question is strictly intrinsic disorder, excluding it is defensible. If the question is broader concept conflict, excluding it removes useful information.

## 10. Export manifest

Run this at the end to list all figures generated by this notebook.

In [ ]:
manifest = pd.DataFrame([
    {'figure': p.name, 'path': str(p.relative_to(ROOT)), 'size_kb': round(p.stat().st_size / 1024, 1)}
    for p in sorted(OUT.glob('*.png'))
])
manifest